# 16 · Python Gotchas Interviewers Love

Twenty-three classic Python surprises, each as **predict → run → why → fix**. After this notebook you can survive the "what does this print?" round and spot these bugs in a code review — several of them are real LLM-app bugs (chat history leaking between users, a `temperature=0` silently becoming 0.7, a generator of documents that is empty the second time).

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [06 · Functions, args, scope and lambda](06_functions_args_scope_lambda.ipynb), [03 · Lists and tuples](03_lists_and_tuples.ipynb)

**How to use it:** for each group, read the snippets in the code cell and write down what you expect *before* running it. Then read the "why" cell. Every surprising claim is checked with an `assert`.

## Why this matters in interviews

- "What does this print?" snippets are a fast way for an interviewer to check that you understand Python's object model — names vs objects, mutability, evaluation time.
- The same gotchas appear as "find the bug" exercises and in code review questions; naming the bug *and* the idiomatic fix is what scores.
- Several are production bugs in AI code: shared mutable state across requests, falsy-zero defaults, exhausted generators, NaN scores in evals.

## What interviewers ask

- "What does this function return the second time you call it?" (mutable default arguments)
- "Why do all the lambdas return the same value?" (late binding)
- "Is `a is b` true here? Why is `0.1 + 0.2 != 0.3`?"
- "Why did changing one row change every row?" (`[[]] * n`, shallow copies)
- "Why can't I put this object in a set?" (`__eq__` without `__hash__`)
- "Spot the bug" in a 10-line snippet: `UnboundLocalError`, a shadowed builtin, a swallowed exception, `x = x.sort()`.

## A. Functions and scope

**Predict** the output of each numbered line below.

1. A default argument that is a list.
2. Lambdas created in a loop.
3. Incrementing a global counter inside a function.
4. Naming a variable `list`.

In [ ]:
# 1. mutable default argument - a chat history helper
def add_message(text, history=[]):
    history.append(text)
    return history

print("1.", add_message("hi from user A"), add_message("hi from user B"))

# 2. late-binding closures - one handler per tool, built in a loop
handlers = [lambda: f"calling {name}" for name in ["search", "weather", "calculator"]]
print("2.", [h() for h in handlers])

# 3. UnboundLocalError
call_count = 0
def track_call():
    call_count += 1
try:
    track_call()
except UnboundLocalError as e:
    print("3.", type(e).__name__ + ":", e)

# 4. shadowing a builtin
list = ["refunds.md", "shipping.md"]            # innocent-looking variable name...
try:
    print(list("abc"))
except TypeError as e:
    print("4.", type(e).__name__ + ":", e)
finally:
    del list                                    # restore the builtin for the rest of the notebook

assert add_message.__defaults__[0] == ["hi from user A", "hi from user B"]
assert [h() for h in handlers] == ["calling calculator"] * 3

**Why:**

1. **Default values are evaluated once, when `def` runs**, and stored on the function (`add_message.__defaults__`). Every call without `history` shares the *same* list — so user B sees user A's message. In a web service that is a data leak between users.
2. **Closures capture variables, not values.** All three lambdas look up `name` when they are *called*, and by then the loop has finished with `name == "calculator"`.
3. Assigning to a name anywhere in a function makes it **local for the whole function**. `call_count += 1` reads the local before it has a value.
4. `list = [...]` rebinds the builtin name in your module, so `list("abc")` calls your list object. In a notebook this persists until you `del` it or restart the kernel. Same trap with `id`, `type`, `input`, `max`, `sum`, `dict`, `str`.

**Fixes:**

In [ ]:
def add_message(text, history=None):            # 1. None sentinel, new list per call
    history = [] if history is None else history
    history.append(text)
    return history

handlers = [lambda name=name: f"calling {name}" for name in ["search", "weather"]]   # 2. bind now via a default
from functools import partial
handlers_partial = [partial("calling {}".format, name) for name in ["search", "weather"]]

class CallCounter:                              # 3. keep state in an object (or return values), not a global
    def __init__(self):
        self.count = 0
    def track(self):
        self.count += 1

counter = CallCounter(); counter.track(); counter.track()
doc_list = ["refunds.md"]                       # 4. just pick another name

print(add_message("A"), add_message("B"), [h() for h in handlers], [h() for h in handlers_partial], counter.count)
assert add_message("B") == ["B"] and [h() for h in handlers] == ["calling search", "calling weather"]
assert list("ab") == ["a", "b"]                 # the builtin is back

## B. Identity, equality and numbers

5. `is` on integers built at runtime.
6. Adding floats.
7. NaN.
8. Chained comparisons.
9. Negative floor division and modulo.

In [ ]:
# 5. `is` vs `==`, and small-int caching
a, b = int("256"), int("256")
c, d = int("257"), int("257")
print("5.", a == b, a is b, "|", c == d, c is d)

# 6. float equality
total = 0.0
for _ in range(10):
    total += 0.1
print("6.", 0.1 + 0.2 == 0.3, 0.1 + 0.2, "| ten `+= 0.1`:", total, "| sum([0.1] * 10):", sum([0.1] * 10))

# 7. NaN is not equal to anything, including itself
nan = float("nan")
print("7.", nan == nan, "| max([nan, 1, 2]) =", max([nan, 1, 2]), "| max([1, nan, 2]) =", max([1, nan, 2]),
      "| nan in [nan]:", nan in [nan])

# 8. chained comparisons
print("8.", 3 > 2 > 1, (3 > 2) > 1, "|", False == False in [False], "|", "a" in "abc" == True)

# 9. floor division and modulo with negatives
print("9.", -7 // 2, -7 % 2, int(-7 / 2), divmod(-7, 2), "| round(2.5) =", round(2.5), "round(3.5) =", round(3.5))

assert total != 1.0 and (a is b) and not (c is d) and not (nan == nan) and max([1, nan, 2]) == 2 and max([nan, 1, 2]) != 2
assert ("a" in "abc" == True) is False and -7 // 2 == -4 and round(2.5) == 2

**Why:**

5. `is` asks "the **same object**?"; `==` asks "equal **value**?". CPython pre-creates the integers −5 to 256, so every 256 is the same object, while each runtime 257 is new. (Literals in one cell may share a constant, so `x = 257; y = 257; x is y` can be `True` — which is exactly why you never rely on it.) Use `is` only for `None`, `True`/`False` and sentinel objects.
6. Floats are binary fractions; `0.1` cannot be represented exactly, so tiny errors appear and accumulate in a `+=` loop. (Since Python 3.12 the built-in `sum()` uses compensated summation for floats, so `sum([0.1] * 10)` happens to give `1.0` — on 3.11 it gives `0.9999999999999999`. Don't rely on either.)
7. NaN compares unequal (and not-less, not-greater) to everything, so `max`, `sorted` and `min` give **order-dependent** results; `nan in [nan]` is `True` only because containers check identity first. An eval with one NaN score can silently change your "best model".
8. `a op1 b op2 c` means `(a op1 b) and (b op2 c)`. So `"a" in "abc" == True` is `("a" in "abc") and ("abc" == True)` → `False`. (`in` and `==` chain too.)
9. `//` floors towards −∞ (C and Java truncate towards 0), and `%` takes the sign of the divisor — handy: `(i - 1) % n` wraps correctly in a ring buffer. `round()` uses banker's rounding (half to even).

**Fixes:**

In [ ]:
import math

x = int("257")
print("5. compare values with ==:", x == 257, "| is only for None:", None is None)
print("6. math.isclose:", math.isclose(0.1 + 0.2, 0.3), "| math.fsum:", math.fsum([0.1] * 10) == 1.0,
      "| in tests: abs(a - b) < 1e-9 or pytest.approx")
scores = [0.71, nan, 0.93, 0.88]
clean = [s for s in scores if not math.isnan(s)]
print("7. drop/flag NaN first:", max(clean), "| count of NaN:", sum(math.isnan(s) for s in scores))
print("8. write what you mean:", "a" in "abc", "| use parentheses when mixing:", ("a" in "abc") == True)
print("9. truncate toward zero explicitly:", math.trunc(-7 / 2), "| C-style remainder:", math.fmod(-7, 2),
      "| ring index:", (0 - 1) % 5)
assert math.isclose(0.1 + 0.2, 0.3) and max(clean) == 0.93

## C. Lists and copying

10. Removing items while looping over the same list.
11. Copying a list of lists.
12. Building a grid with `*`.
13. Saving the result of `.sort()`.

In [ ]:
# 10. modifying a list while iterating over it
nums = [1, 2, 2, 3, 4]
for n in nums:
    if n % 2 == 0:
        nums.remove(n)
print("10.", nums)

# 11. shallow copy of nested lists - two prompt variants sharing a message list
base = [["system", "Be concise."], ["user", "Summarise the doc."]]
variant = base.copy()                          # or list(base) or base[:]
variant[0][1] = "Be verbose."
print("11.", base[0])

# 12. [[]] * n - one row per retrieved chunk
grid = [[]] * 3
grid[0].append("chunk-A")
print("12.", grid)

# 13. sort() returns None
scores = [0.3, 0.9, 0.5]
top = scores.sort(reverse=True)
print("13.", top, "| but scores is now", scores)

assert nums == [1, 2, 3] and base[0][1] == "Be verbose." and grid == [["chunk-A"]] * 3 and top is None

**Why:**

10. The loop walks by **index**. Removing the first `2` shifts everything left, so the loop's next index skips the second `2`. (Doing this to a dict raises `RuntimeError: dictionary changed size during iteration`.)
11. `.copy()`, `list(x)` and `x[:]` are **shallow**: a new outer list holding the *same* inner lists. Change an inner list through one and you see it through both.
12. `[[]] * 3` repeats a **reference** to one inner list three times.
13. Methods that mutate in place — `list.sort`, `list.reverse`, `list.append`, `random.shuffle`, `dict.update` — return `None` by design, so you cannot mistake them for "a sorted copy".

**Fixes:**

In [ ]:
import copy

nums = [1, 2, 2, 3, 4]
evens_removed = [n for n in nums if n % 2]                  # 10. build a new list (or loop over nums[:])
base = [["system", "Be concise."], ["user", "Summarise the doc."]]
variant = copy.deepcopy(base)                               # 11. deep copy nested structures
variant[0][1] = "Be verbose."
grid = [[] for _ in range(3)]                                # 12. a new inner list per row
grid[0].append("chunk-A")
scores = [0.3, 0.9, 0.5]
top = sorted(scores, reverse=True)                           # 13. sorted() returns a new list
print("10.", evens_removed, "| 11.", base[0], "| 12.", grid, "| 13.", top, scores)
assert evens_removed == [1, 3] and base[0][1] == "Be concise." and grid == [["chunk-A"], [], []] and top == [0.9, 0.5, 0.3]

## D. Strings, booleans and tuples

14. Building a big string with `+=` in a loop.
15. What `and` / `or` actually return.
16. Parentheses that do not make a tuple.

In [ ]:
# 15. `or` returns an operand, not True/False - a very common config bug
def make_request(temperature=None):
    temperature = temperature or 0.7            # "default to 0.7 if not given"
    return {"temperature": temperature}

print("15.", "" or "anonymous", "|", 0 and 1 / 0, "|", [] or None, "|", make_request(temperature=0))

# 16. one-element tuples
stop = ("\n\n")                                  # meant: a tuple of stop sequences
ids = (42,)
also_tuple = 5,
print("16.", type(stop).__name__, [s for s in ("END")], "|", type(ids).__name__, type(also_tuple).__name__)

assert make_request(temperature=0) == {"temperature": 0.7} and isinstance(stop, str) and also_tuple == (5,)

**Why:**

15. `x or y` returns `x` if `x` is truthy, else `y`; `x and y` returns `x` if it is falsy, else `y`. Both short-circuit (`0 and 1/0` never divides). The config bug: `0` is falsy, so an explicit `temperature=0` silently becomes `0.7`. Falsy values: `0`, `0.0`, `""`, `[]`, `{}`, `set()`, `None`, `False`.
16. The **comma** makes a tuple, not the parentheses. `("END")` is just a string, and iterating a string gives characters — so an API expecting a sequence of stop strings gets `["E", "N", "D"]`. A stray trailing comma (`x = 5,`) makes a tuple by accident.

**14 — string `+=` in a loop.** Strings are immutable, so `s += piece` builds a new string and copies everything so far: quadratic in general. CPython has a special case that extends the string in place when nothing else refers to it, so simple loops often look fine — but it is an implementation detail (not in PyPy, and it disappears as soon as a second reference exists). The chart shows all three cases; `"".join(parts)` is linear everywhere.

In [ ]:
import time

import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "axes.axisbelow": True})

def plus_equals(parts):
    s = ""
    for p in parts:
        s += p
    return s

def plus_equals_with_alias(parts):
    s = ""
    for p in parts:
        alias = s                                # a second reference disables CPython's in-place trick
        s += p
    return s

def join(parts):
    return "".join(parts)

def best_time(fn, parts, repeat=3):
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter(); fn(parts); best = min(best, time.perf_counter() - start)
    return best * 1000

sizes = [2000, 4000, 6000, 8000, 10000]
results = {fn.__name__: [best_time(fn, [f"chunk{i:05d} " for i in range(n)]) for n in sizes]
           for fn in (plus_equals, plus_equals_with_alias, join)}
fig, ax = plt.subplots(figsize=(8.5, 3.8))
for (name, ms), color in zip(results.items(), [AQUA, ORANGE, BLUE]):
    ax.plot(sizes, ms, marker="o", lw=2, color=color, label=name)
ax.set_xlabel("number of pieces joined"); ax.set_ylabel("milliseconds (best of 3)")
ax.set_title("s += piece is quadratic unless CPython's in-place trick applies; join is linear")
ax.legend(frameon=False)
plt.show()
print({name: f"{ms[-1]:.1f} ms" for name, ms in results.items()}, "at", sizes[-1], "pieces")
assert plus_equals(["a", "b"]) == plus_equals_with_alias(["a", "b"]) == join(["a", "b"]) == "ab"
assert results["plus_equals_with_alias"][-1] > 3 * results["join"][-1]

**Fixes:**

In [ ]:
def make_request(temperature=None):
    temperature = 0.7 if temperature is None else temperature     # 15. test for None, not falsiness
    return {"temperature": temperature}

stop = ("\n\n",)                                                  # 16. the trailing comma makes the tuple
parts = [f"line {i}" for i in range(3)]
text = "\n".join(parts)                                           # 14. collect pieces, join once
print("15.", make_request(temperature=0), "| 16.", stop, "| 14.", repr(text))
assert make_request(temperature=0) == {"temperature": 0} and isinstance(stop, tuple)

## E. Iterators, classes and exceptions

17. Using a generator twice.
18. A class with `__eq__` in a set.
19. A list defined on the class.
20. A "safe" `try/except` around a calculation.

In [ ]:
# 17. generator exhaustion - a lazy stream of retrieved documents
docs = (f"doc{i}" for i in range(3))
print("17.", "logged", len(list(docs)), "docs; then embedded", len(list(docs)), "docs")

# 18. __eq__ without __hash__
class Chunk:
    def __init__(self, text):
        self.text = text
    def __eq__(self, other):
        return isinstance(other, Chunk) and self.text == other.text
try:
    unique = {Chunk("a"), Chunk("a")}
except TypeError as e:
    print("18.", type(e).__name__ + ":", e)

# 19. class attribute shared by every instance
class Conversation:
    messages = []                               # looks like a per-object default...
    def add(self, text):
        self.messages.append(text)
alice, bob = Conversation(), Conversation()
alice.add("my account number is 1234")
print("19. bob sees:", bob.messages)

# 20. broad except hides a bug
def weighted_score(record, weight=2.0):
    try:
        return float(record["score"]) * weigth  # typo -> NameError
    except Exception:
        return 0.0
print("20.", [weighted_score({"score": s}) for s in ["0.9", "0.4"]])

assert Chunk.__hash__ is None and bob.messages == ["my account number is 1234"]

**Why:**

17. A generator (also `map`, `filter`, `zip`, an open file) is a **one-pass** iterator. The first `list(docs)` consumed it; the second sees nothing — no error, just an empty pipeline.
18. Defining `__eq__` makes Python set `__hash__ = None` (objects that are equal must hash equally, and Python cannot guess your hash), so instances become **unhashable**: no sets, no dict keys.
19. `messages = []` in the class body is **one list stored on the class**. `self.messages.append` finds it through the class and mutates it for everyone — another cross-user leak.
20. `except Exception` also catches `NameError`, `AttributeError` and `TypeError` — the errors that mean *your code is wrong*. Every score is silently 0.0.

**Fixes:**

In [ ]:
from dataclasses import dataclass, field

docs = [f"doc{i}" for i in range(3)]                  # 17. materialise if you need two passes (or re-create)
print("17.", len(docs), len(docs))

@dataclass(frozen=True)                               # 18. frozen dataclass: __eq__ AND a matching __hash__
class Chunk:
    text: str
print("18.", {Chunk("a"), Chunk("a")})

@dataclass
class Conversation:                                   # 19. per-instance state, created in __init__
    messages: list[str] = field(default_factory=list)
    def add(self, text):
        self.messages.append(text)
alice, bob = Conversation(), Conversation()
alice.add("my account number is 1234")
print("19. bob sees:", bob.messages)

def weighted_score(record, weight=2.0):
    try:
        raw = float(record["score"])
    except (KeyError, ValueError):                    # 20. only the failures we expect from the data
        return 0.0
    return raw * weight                               # a typo here would now crash loudly
print("20.", [weighted_score({"score": s}) for s in ["0.9", "0.4"]])
assert len({Chunk("a"), Chunk("a")}) == 1 and bob.messages == [] and weighted_score({"score": "0.9"}) == 1.8

## F. Ordering, time and recursion

21. The order you get back from a dict and from a set.
22. Comparing two timestamps.
23. Walking a deeply nested structure recursively.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone

# 21. dicts keep insertion order; sets do not promise any order (and it varies between runs)
print("21. dict:", list({"cherry": 1, "apple": 2, "banana": 3}))
cmd = "print(list({'apple', 'banana', 'cherry', 'date'}))"
orders = [subprocess.run([sys.executable, "-c", cmd], capture_output=True, text=True,
                         env={**os.environ, "PYTHONHASHSEED": seed}).stdout.strip() for seed in ("1", "2")]
print("    same set in two fresh processes:", orders[0], "vs", orders[1])

# 22. naive vs aware datetimes
cached_at = datetime(2026, 9, 29, 9, 0)                        # naive: no timezone
now = datetime(2026, 9, 29, 9, 30, tzinfo=timezone.utc)        # aware
try:
    print(now - cached_at)
except TypeError as e:
    print("22.", type(e).__name__ + ":", e)

# 23. the default recursion limit
def depth(node):
    return 1 + max((depth(child) for child in node.get("children", [])), default=0)

tree = node = {}
for _ in range(5000):                                          # a 5,000-level nested JSON-like tree
    node["children"] = [{}]
    node = node["children"][0]
try:
    depth(tree)
except RecursionError as e:
    print("23.", type(e).__name__ + ":", str(e)[:60], "| limit:", sys.getrecursionlimit())
try:
    json.loads("[" * 100_000 + "]" * 100_000)
except RecursionError:
    print("    json.loads on 100,000 nested brackets -> RecursionError too")
assert orders[0] != orders[1]

**Why:**

21. Since Python 3.7, **dicts preserve insertion order** — guaranteed. **Sets** are ordered by hash, and string hashes are randomised per process (`PYTHONHASHSEED`), so the same code prints a different order on each run — a classic source of flaky tests and non-reproducible prompts (e.g. building a few-shot prompt from a set).
22. A naive datetime has no timezone, so Python refuses to compare or subtract it from an aware one. Worse, `naive.timestamp()` assumes the *machine's* local timezone, so the same code gives different answers on your laptop and on a UTC server.
23. CPython limits recursion depth (1,000 by default) to protect the C stack. Real data — nested JSON from a tool, a deep document tree, a long linked chain — can exceed it; deeply nested JSON is even a denial-of-service trick against parsers.

**Fixes:**

In [ ]:
labels = {"refund", "billing", "shipping"}
print("21. sort sets before showing or prompting:", sorted(labels), "| dedupe keeping order:", list(dict.fromkeys(["b", "a", "b"])))

cached_at = datetime(2026, 9, 29, 9, 0, tzinfo=timezone.utc)   # 22. aware UTC everywhere
print("22.", now - cached_at, "| now():", type(datetime.now(timezone.utc).tzinfo).__name__)

def depth_iterative(root):                                     # 23. explicit stack instead of recursion
    best, stack = 0, [(root, 1)]
    while stack:
        node, d = stack.pop()
        best = max(best, d)
        stack.extend((child, d + 1) for child in node.get("children", []))
    return best
print("23. iterative depth:", depth_iterative(tree), "(raising sys.setrecursionlimit only moves the crash)")
assert depth_iterative(tree) == 5001 and str(now - cached_at) == "0:30:00"

## Try it yourself

**1.** Spot the bug: `dedupe` should remove duplicates but **keep the first-seen order** of retrieved chunk ids. Fix it.

In [ ]:
# Solution — try it yourself first, then run this
def dedupe_buggy(ids):
    return list(set(ids))                      # loses order (and the order changes between runs)

def dedupe(ids):
    return list(dict.fromkeys(ids))            # dicts keep insertion order

ids = ["c7", "a1", "c7", "b3", "a1"]
print(dedupe(ids))
assert dedupe(ids) == ["c7", "a1", "b3"]

**2.** Spot the bug: `top_k` crashes with `TypeError: 'NoneType' object is not subscriptable`, and after fixing that it also changes the caller's list. Fix both.

In [ ]:
# Solution — try it yourself first, then run this
def top_k_buggy(scores, k=2):
    return scores.sort(reverse=True)[:k]       # .sort() returns None and mutates the input

def top_k(scores, k=2):
    return sorted(scores, reverse=True)[:k]

scores = [0.2, 0.9, 0.5]
print(top_k(scores), scores)
assert top_k(scores) == [0.9, 0.5] and scores == [0.2, 0.9, 0.5]

**3.** Spot the bug: the callbacks should multiply by 1, 2 and 3, so for `x = 10` you expect `[10, 20, 30]`.

In [ ]:
# Solution — try it yourself first, then run this
callbacks_buggy = [lambda x: x * i for i in range(1, 4)]            # all use the final i == 3
callbacks = [lambda x, i=i: x * i for i in range(1, 4)]             # bind i at definition time
print([f(10) for f in callbacks_buggy], "->", [f(10) for f in callbacks])
assert [f(10) for f in callbacks] == [10, 20, 30]

## One-page summary

| # | Gotcha | The surprise | The fix |
|---|---|---|---|
| 1 | mutable default argument | `def f(x=[])` shares one list across calls | `x=None`, then `x = [] if x is None else x` |
| 2 | late-binding closures | lambdas in a loop all see the last value | `lambda i=i: ...` or `functools.partial` |
| 3 | `UnboundLocalError` | assigning a name makes it local for the whole function | return values, an object, or `global`/`nonlocal` |
| 4 | shadowing builtins | `list = [...]` breaks `list()` | never reuse `list`, `dict`, `id`, `type`, `input`, `sum`... |
| 5 | `is` vs `==` | small ints are cached, others are not | `==` for values; `is` only for `None`/sentinels |
| 6 | float equality | `0.1 + 0.2 != 0.3` | `math.isclose`, `math.fsum`, `Decimal` for money |
| 7 | NaN | `nan != nan`; `max` depends on position | `math.isnan`; drop or flag NaNs first |
| 8 | chained comparisons | `"a" in "abc" == True` is `False` | don't compare to `True`; add parentheses |
| 9 | negative `//` and `%` | `-7 // 2 == -4`, `-7 % 2 == 1`, `round(2.5) == 2` | `math.trunc`, `math.fmod`; know floor semantics |
| 10 | mutating while iterating | skipped elements; dicts raise | build a new list or iterate a copy |
| 11 | shallow copy | inner lists are shared | `copy.deepcopy` |
| 12 | `[[]] * n` | n references to one list | `[[] for _ in range(n)]` |
| 13 | `.sort()` returns `None` | `x = x.sort()` loses your list | `sorted(x)` for a new list |
| 14 | string `+=` in a loop | quadratic copying (unless CPython's trick applies) | `"".join(parts)` |
| 15 | `and` / `or` return operands | `temperature or 0.7` turns 0 into 0.7 | `0.7 if t is None else t` |
| 16 | one-element tuple | `("END")` is a string | `("END",)` — the comma makes the tuple |
| 17 | generator exhaustion | second pass is silently empty | materialise with `list()` or re-create |
| 18 | `__eq__` without `__hash__` | instances become unhashable | define `__hash__` too, or `@dataclass(frozen=True)` |
| 19 | class attribute shared mutable | one list for every instance | create it in `__init__` / `field(default_factory=list)` |
| 20 | broad `except` | bugs become silent default values | catch specific exceptions; log the rest |
| 21 | set order | varies between processes (hash randomisation) | `sorted(...)`; `dict.fromkeys` to dedupe in order |
| 22 | naive vs aware datetimes | `TypeError`, or wrong timestamps on other machines | aware UTC everywhere: `datetime.now(timezone.utc)` |
| 23 | recursion limit | `RecursionError` at ~1,000 levels | iterate with an explicit stack |

## Say it in an interview

- **The one idea behind half of these:** "Python names are references to objects. Mutating an object is visible through every name that refers to it — default arguments, class attributes, shallow copies and `[[]] * n` are all that one fact."
- **Evaluation time:** "Default arguments are evaluated once at `def` time; closures look variables up at call time."
- **Identity vs equality:** "`is` compares identity, `==` compares value; I use `is` only for `None` and sentinels."
- **Truthiness:** "`or` returns an operand, so `x or default` is wrong whenever `0`, `""` or `[]` is a valid value — test `is None`."
- **Traps worth naming in review:** a shared mutable default in a request handler, `except Exception: pass`, iterating a generator twice, NaN in a metric, sets in anything that must be reproducible.

## Next

- [01 · Warm-up problems](../02_python_problems/01_warmup_problems.ipynb) — put the fundamentals to work
- [Big-O and timing](../03_dsa/01_big_o_and_timing.ipynb) — why `join` beats `+=`, and `set` beats `list` for lookups
- Back to the foundations: [06 · Functions, args, scope and lambda](06_functions_args_scope_lambda.ipynb) · [07 · Comprehensions, iterators and generators](07_comprehensions_iterators_generators.ipynb) · [08 · Classes, OOP and dataclasses](08_classes_oop_dataclasses.ipynb)
- Theory: [python_basics course](../../python_basics/index.html) · [interview bank](../../ai_interview_prep/index.html)